# EDA & Data Cleaning
Track **exactly** what changes at every step — row counts, value counts, before/after distributions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('whitegrid')

train = pd.read_csv('../../dataset/train.csv')
test  = pd.read_csv('../../dataset/test.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
train.head()

## 1. Schema & Basic Info

In [ ]:
train.dtypes.to_frame('dtype').assign(
    n_unique    = train.nunique(),
    n_missing   = train.isnull().sum(),
    missing_pct = (train.isnull().mean() * 100).round(2)
)

In [ ]:
train.describe(include='all')

## 2. Missing Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in [(axes[0], train, 'Train'), (axes[1], test, 'Test')]:
    miss = (df.isnull().mean() * 100).sort_values(ascending=False)
    miss = miss[miss > 0]
    bars = ax.bar(miss.index, miss.values, color='steelblue')
    ax.set_title(f'{title}: Missing % per Column')
    ax.set_ylabel('Missing %')
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, miss.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.05, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('=== Train missing ===')
print(train.isnull().sum()[train.isnull().sum() > 0])
print('\n=== Test missing ===')
print(test.isnull().sum()[test.isnull().sum() > 0])

## 3. Target Variable — `demand`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(train['demand'], bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Demand — Raw Distribution')
axes[0].set_xlabel('demand')

axes[1].hist(np.log1p(train['demand']), bins=80, color='seagreen', edgecolor='none')
axes[1].set_title('log1p(demand)')

train['demand'].plot(kind='box', ax=axes[2])
axes[2].set_title('Demand Boxplot')

plt.tight_layout()
plt.show()

print(train['demand'].describe())
print(f'\ndemand < 1e-4  : {(train["demand"] < 1e-4).sum():>5}  (near-zero — these are real empty-road readings, keep them)')
print(f'demand == 1.0  : {(train["demand"] == 1.0).sum():>5}  (max demand  — all on highways, keep them)')

# Confirm demand==1.0 are all highways
print('\nRoadType where demand==1.0:')
print(train[train['demand']==1.0]['RoadType'].value_counts(dropna=False))

## 4. Categorical Features

In [ ]:
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for i, col in enumerate(cat_cols):
    vc = train[col].value_counts(dropna=False)
    x_labels = [str(v) if pd.notna(v) else 'NaN' for v in vc.index]
    axes[0][i].bar(x_labels, vc.values, color='steelblue')
    axes[0][i].set_title(f'{col}\nValue Counts')
    axes[0][i].tick_params(axis='x', rotation=30)

    tmp = train[[col, 'demand']].copy()
    tmp[col] = tmp[col].fillna('Missing')
    tmp.boxplot(column='demand', by=col, ax=axes[1][i])
    axes[1][i].set_title(f'Demand by {col}')
    axes[1][i].set_xlabel('')
    axes[1][i].tick_params(axis='x', rotation=30)

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-field consistency checks
print('=== RoadType × LargeVehicles ===')
print(pd.crosstab(train['RoadType'], train['LargeVehicles'], dropna=False))

print('\n=== RoadType × NumberofLanes (stats) ===')
print(train.groupby('RoadType')['NumberofLanes'].describe())

# Weather vs Temperature contradictions
snowy_hot  = train[(train['Weather']=='Snowy') & (train['Temperature'] > 10)]
sunny_cold = train[(train['Weather']=='Sunny') & (train['Temperature'] < 0)]
print(f'\nContradictions — Snowy+Temp>10 : {len(snowy_hot)}')
print(f'Contradictions — Sunny+Temp<0  : {len(sunny_cold)}')

## 5. Temperature — Distribution & Outliers

In [ ]:
q1  = train['Temperature'].quantile(0.25)
q3  = train['Temperature'].quantile(0.75)
iqr = q3 - q1
lo_15 = q1 - 1.5 * iqr   # standard Tukey fence
hi_15 = q3 + 1.5 * iqr
lo_3  = q1 - 3.0 * iqr   # far fence
hi_3  = q3 + 3.0 * iqr

out_15 = train['Temperature'].notna() & ((train['Temperature'] < lo_15) | (train['Temperature'] > hi_15))
out_3  = train['Temperature'].notna() & ((train['Temperature'] < lo_3)  | (train['Temperature'] > hi_3))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(train['Temperature'].dropna(), bins=80, color='tomato', edgecolor='none')
axes[0].axvline(lo_15, color='red',    linestyle='--', label=f'1.5×IQR lo={lo_15:.1f}')
axes[0].axvline(hi_15, color='red',    linestyle='--', label=f'1.5×IQR hi={hi_15:.1f}')
axes[0].axvline(lo_3,  color='orange', linestyle=':',  label=f'3×IQR lo={lo_3:.1f}')
axes[0].axvline(hi_3,  color='orange', linestyle=':',  label=f'3×IQR hi={hi_3:.1f}')
axes[0].set_title('Temperature + IQR Fences')
axes[0].legend(fontsize=8)

axes[1].hist(train.loc[out_15, 'Temperature'], bins=30, color='tomato', edgecolor='none')
axes[1].set_title(f'Outlier values ({out_15.sum()} rows outside 1.5×IQR fence)')
axes[1].set_xlabel('Temperature')

# Outliers by weather type
weather_out = train.loc[out_15, 'Weather'].value_counts(dropna=False)
axes[2].bar([str(v) if pd.notna(v) else 'NaN' for v in weather_out.index], weather_out.values, color='coral')
axes[2].set_title('Outlier rows by Weather type')
axes[2].set_xlabel('Weather')

plt.tight_layout()
plt.show()

print(f'Temperature IQR       : {iqr:.2f}')
print(f'1.5×IQR fence         : [{lo_15:.2f}, {hi_15:.2f}]  →  {out_15.sum()} outlier values')
print(f'3×IQR fence           : [{lo_3:.2f},  {hi_3:.2f}]   →  {out_3.sum()} outlier values  ← (was used before, catches 0!)')
print(f'\nOutlier weather breakdown:')
print(train.loc[out_15, 'Weather'].value_counts(dropna=False).to_string())
print('\n→ Hot=Sunny, Cold=Snowy — no contradictions. We CLIP these, not drop.')

## 6. Timestamp & Day

In [ ]:
def to_minutes(ts_series):
    parts = ts_series.str.split(':', expand=True).astype(int)
    return parts[0] * 60 + parts[1]

train['_min'] = to_minutes(train['timestamp'])

print('Days in train :', sorted(train['day'].unique()))
print('Time range    :', train['_min'].min(), '—', train['_min'].max(), 'minutes')
print('Unique slots  :', train['_min'].nunique())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hourly = train.groupby('_min')['demand'].mean()
axes[0].plot(hourly.index / 60, hourly.values, color='steelblue', linewidth=1.5)
axes[0].set_title('Mean Demand by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Mean Demand')

day_demand = train.groupby('day')['demand'].mean()
axes[1].bar(day_demand.index.astype(str), day_demand.values, color='seagreen')
axes[1].set_title('Mean Demand by Day')
axes[1].set_xlabel('Day')

plt.tight_layout()
plt.show()

train.drop(columns=['_min'], inplace=True)

## 7. Geohash Coverage

In [ ]:
train_geo = set(train['geohash'])
test_geo  = set(test['geohash'])

print(f'Unique geohashes — train: {len(train_geo)}   test: {len(test_geo)}')
print(f'Only in train : {len(train_geo - test_geo)}')
print(f'Only in test  : {len(test_geo - train_geo)}')
print(f'In both       : {len(train_geo & test_geo)}')

geo_demand = train.groupby('geohash')['demand'].mean().sort_values(ascending=False)
print('\nTop 10 geohashes by mean demand:')
print(geo_demand.head(10).to_string())

## 8. Correlation Heatmap

In [ ]:
num_cols = ['demand', 'Temperature', 'NumberofLanes']
plt.figure(figsize=(6, 4))
sns.heatmap(train[num_cols].corr(), annot=True, fmt='.3f', cmap='coolwarm', center=0, square=True)
plt.title('Numerical Feature Correlations')
plt.tight_layout()
plt.show()

---
# Data Cleaning

| # | Step | What changes |
|---|------|--------------|
| 1 | Drop raw `Index` column | structural |
| 2 | Drop exact duplicates | rows removed |
| 3 | Impute `RoadType` (mode per geohash) | 600 train / 324 test values filled |
| 4 | Impute `Weather` (mode per geohash+timestamp) | 797 train / 431 test values filled |
| 5 | Impute `Temperature` (median per geohash+timestamp+weather) | 2495 train / 1349 test values filled |
| 6 | **Clip Temperature at 1.5×IQR** | 555 train / ~26 test values capped |


In [ ]:
df_train = train.copy()
df_test  = test.copy()
print(f'Start — train: {df_train.shape}  test: {df_test.shape}')

### Step 1 — Drop raw `Index` column

In [ ]:
df_train.drop(columns=['Index'], inplace=True, errors='ignore')
df_test.drop(columns=['Index'],  inplace=True, errors='ignore')
print('Columns:', df_train.columns.tolist())

### Step 2 — Drop exact duplicates

In [ ]:
before = len(df_train)
df_train.drop_duplicates(inplace=True)
df_train.reset_index(drop=True, inplace=True)
print(f'Exact duplicates removed: {before - len(df_train)}   (train rows now: {len(df_train)})')

# Check key duplicates — same location at same time should be unique
key_dupes = df_train.duplicated(subset=['geohash', 'day', 'timestamp']).sum()
print(f'Key duplicates (geohash+day+timestamp): {key_dupes}')

### Step 3 — Impute `RoadType`
Strategy: use the most common road type for that geohash. If the geohash is unseen, fall back to global mode ('Residential').

In [ ]:
before_train = df_train['RoadType'].isnull().sum()
before_test  = df_test['RoadType'].isnull().sum()

geo_road_mode = (
    df_train.dropna(subset=['RoadType'])
            .groupby('geohash')['RoadType']
            .agg(lambda x: x.mode()[0])
)
global_road_mode = df_train['RoadType'].mode()[0]

for df in [df_train, df_test]:
    mask = df['RoadType'].isnull()
    df.loc[mask, 'RoadType'] = df.loc[mask, 'geohash'].map(geo_road_mode).fillna(global_road_mode)

print(f'RoadType missing filled — train: {before_train} → {df_train["RoadType"].isnull().sum()}')
print(f'RoadType missing filled — test : {before_test}  → {df_test["RoadType"].isnull().sum()}')
print('\nTrain RoadType distribution after fill:')
print(df_train['RoadType'].value_counts())

### Step 4 — Impute `Weather`
Strategy: mode per (geohash, timestamp). Fall back to global mode.

In [ ]:
before_train = df_train['Weather'].isnull().sum()
before_test  = df_test['Weather'].isnull().sum()

geo_ts_weather = (
    df_train.dropna(subset=['Weather'])
            .groupby(['geohash', 'timestamp'])['Weather']
            .agg(lambda x: x.mode()[0])
)
global_weather_mode = df_train['Weather'].mode()[0]

for df in [df_train, df_test]:
    mask = df['Weather'].isnull()
    lookup_key = list(zip(df.loc[mask, 'geohash'], df.loc[mask, 'timestamp']))
    df.loc[mask, 'Weather'] = [geo_ts_weather.get(k, global_weather_mode) for k in lookup_key]

print(f'Weather missing filled — train: {before_train} → {df_train["Weather"].isnull().sum()}')
print(f'Weather missing filled — test : {before_test}  → {df_test["Weather"].isnull().sum()}')
print('\nTrain Weather distribution after fill:')
print(df_train['Weather'].value_counts())

### Step 5 — Impute `Temperature`
Strategy: median per (geohash, timestamp, weather) — three-level lookup. Fall back to (geohash, weather) median, then global median.

In [ ]:
before_train = df_train['Temperature'].isnull().sum()
before_test  = df_test['Temperature'].isnull().sum()

# Level-1: (geohash, timestamp, weather) median
lvl1 = (
    df_train.dropna(subset=['Temperature'])
            .groupby(['geohash', 'timestamp', 'Weather'])['Temperature']
            .median()
)
# Level-2: (geohash, weather) median
lvl2 = (
    df_train.dropna(subset=['Temperature'])
            .groupby(['geohash', 'Weather'])['Temperature']
            .median()
)
# Level-3: global median
global_temp_med = df_train['Temperature'].median()

def impute_temp(df):
    mask = df['Temperature'].isnull()
    if mask.sum() == 0:
        return
    rows = df[mask]
    filled = []
    for _, r in rows.iterrows():
        v = lvl1.get((r['geohash'], r['timestamp'], r['Weather']))
        if v is None:
            v = lvl2.get((r['geohash'], r['Weather']))
        if v is None:
            v = global_temp_med
        filled.append(v)
    df.loc[mask, 'Temperature'] = filled

impute_temp(df_train)
impute_temp(df_test)

print(f'Temperature filled — train: {before_train} → {df_train["Temperature"].isnull().sum()}')
print(f'Temperature filled — test : {before_test}  → {df_test["Temperature"].isnull().sum()}')

### Step 6 — Clip Temperature at 1.5×IQR
Fences are computed on the **training set only** (no test leakage). The 555 outlier values are valid (hot sunny / cold snowy) but extreme — we clip rather than drop to preserve row count.

In [ ]:
# Compute fences on training data AFTER imputation
q1  = df_train['Temperature'].quantile(0.25)
q3  = df_train['Temperature'].quantile(0.75)
iqr = q3 - q1
TEMP_LO = q1 - 1.5 * iqr
TEMP_HI = q3 + 1.5 * iqr

before_temp_train = df_train['Temperature'].copy()
before_temp_test  = df_test['Temperature'].copy()

n_clip_train = ((df_train['Temperature'] < TEMP_LO) | (df_train['Temperature'] > TEMP_HI)).sum()
n_clip_test  = ((df_test['Temperature']  < TEMP_LO) | (df_test['Temperature']  > TEMP_HI)).sum()

df_train['Temperature'] = df_train['Temperature'].clip(TEMP_LO, TEMP_HI)
df_test['Temperature']  = df_test['Temperature'].clip(TEMP_LO, TEMP_HI)

print(f'Clip range          : [{TEMP_LO:.2f}, {TEMP_HI:.2f}]')
print(f'Values clipped train: {n_clip_train}')
print(f'Values clipped test : {n_clip_test}')

# Before / After plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(before_temp_train.dropna(), bins=80, color='tomato',    alpha=0.7, label='Before')
axes[0].hist(df_train['Temperature'],    bins=80, color='steelblue', alpha=0.7, label='After')
axes[0].axvline(TEMP_LO, color='red', linestyle='--', label=f'lo={TEMP_LO:.1f}')
axes[0].axvline(TEMP_HI, color='red', linestyle='--', label=f'hi={TEMP_HI:.1f}')
axes[0].set_title('Train Temperature — Before vs After Clip')
axes[0].legend()

axes[1].hist(before_temp_test.dropna(), bins=80, color='tomato',    alpha=0.7, label='Before')
axes[1].hist(df_test['Temperature'],    bins=80, color='steelblue', alpha=0.7, label='After')
axes[1].set_title('Test Temperature — Before vs After Clip')
axes[1].legend()

plt.tight_layout()
plt.show()

## Final Quality Check

In [ ]:
print('=== Remaining nulls — TRAIN ===')
nulls = df_train.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else 'None ✓')

print('\n=== Remaining nulls — TEST ===')
nulls_t = df_test.isnull().sum()
print(nulls_t[nulls_t > 0] if nulls_t.any() else 'None ✓')

print(f'\nFinal shapes — train: {df_train.shape}  test: {df_test.shape}')
print(f'\nTemperature range — train: [{df_train["Temperature"].min():.2f}, {df_train["Temperature"].max():.2f}]')
print(f'Temperature range — test : [{df_test["Temperature"].min():.2f},  {df_test["Temperature"].max():.2f}]')
print(f'\nDemand range: [{df_train["demand"].min():.8f}, {df_train["demand"].max():.6f}]')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_train['Temperature'], bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Temperature After Cleaning')

axes[1].hist(df_train['demand'], bins=80, color='seagreen', edgecolor='none')
axes[1].set_title('Demand After Cleaning')

for col in ['RoadType', 'Weather']:
    vc = df_train[col].value_counts()
axes[2].bar(df_train['Weather'].value_counts().index,
            df_train['Weather'].value_counts().values, color='coral')
axes[2].set_title('Weather Distribution After Cleaning')

plt.tight_layout()
plt.show()

## Save Cleaned Data

In [ ]:
import os
os.makedirs('../../processed', exist_ok=True)

df_train.to_csv('../../processed/train_cleaned_for_good.csv', index=False)
df_test.to_csv( '../../processed/test_cleaned_for_good.csv',  index=False)

print(f'Saved train: ../../processed/train_cleaned_for_good.csv  {df_train.shape}')
print(f'Saved test : ../../processed/test_cleaned_for_good.csv   {df_test.shape}')
df_train.head()